In [1]:
import os
import random

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from timm.utils import ModelEmaV2
from torch.utils.data import DataLoader
from scipy.io import loadmat
from torchvision import transforms
import torch
import torch.nn as nn
from torchvision import models
from dataset import BreedDataset

/home/danylo/GIT/dog-classifier/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reproducibility

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
DEVICE = 'cuda'

Load Dataset Metadata

In [ ]:
load_dotenv()
DATASET = os.getenv('DATASET_PATH')

mat_data = loadmat(f'{DATASET}/file_list.mat')

data = []

for num, img_path in enumerate(mat_data['file_list']):
    data.append({
        'img_path': f'{DATASET}/images/{str(img_path[0][0])}',
        'annotation_path': f'{DATASET}/annotation/{mat_data['annotation_list'][num][0][0]}',
        'label': mat_data['labels'][num][0],
    })

df = pd.DataFrame(data)

df.head()

Data Augmentations

In [ ]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(480, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((480, 480)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225]),
])


Datasets and DataLoaders

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_dataset = BreedDataset(train_df, transform=train_transform)
val_dataset   = BreedDataset(val_df,   transform=val_transform)
test_dataset  = BreedDataset(test_df,  transform=val_transform)

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=4, pin_memory=True, generator=g)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=4, pin_memory=True, generator=g)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=4, pin_memory=True, generator=g)

Get Model

In [ ]:
def get_model(num_classes=120):
    mdl = models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.DEFAULT)

    for param in mdl.features.parameters():
        param.requires_grad = False

    for param in mdl.features[-4:].parameters():
        param.requires_grad = True

    in_features = mdl.classifier[1].in_features
    mdl.classifier = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(in_features, num_classes))

    return mdl.to(DEVICE)

Train Epoch

In [ ]:
def train_epoch(mdl, loader, criterion, optimizer, ema):
    mdl.train()
    total_loss, correct = 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to('cuda'), labels.to('cuda')

        optimizer.zero_grad()

        outputs = mdl(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        ema.update(mdl)

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


In [ ]:
@torch.no_grad()
def eval_epoch(mdl, loader, criterion):
    mdl.eval()
    total_loss, correct = 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to('cuda'), labels.to('cuda')

        outputs = mdl(imgs)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

Training Loop

In [ ]:
model = get_model()

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam([
    {'params': model.features[-4:-2].parameters(), 'lr': 1e-5},
    {'params': model.features[-2:].parameters(), 'lr': 3e-5},
    {'params': model.classifier.parameters(), 'lr': 3e-4}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
ema = ModelEmaV2(model, decay=0.99)

best_val_acc = 0

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, ema)
    val_loss, val_acc = eval_epoch(ema.module, val_loader, criterion)
    scheduler.step()

    print(
        f'Epoch {epoch+1}/10 '
        f'| train loss: {train_loss:.4f} acc: {train_acc:.4f} '
        f'| val loss: {val_loss:.4f} acc: {val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(ema.module.state_dict(), 'best_model.pth')
        print(f' -> saved best model (val_acc={val_acc:.4f})')